# 🐉 NINE-1 — Treinamento Completo no Kaggle

Treina a **NINE-1**, uma IA de geração de código em **PT-BR** construída do zero em PyTorch.

**Pipeline completo:**
1. ✅ Clona o repositório do GitHub
2. ✅ Processa dados seed (~125 exemplos PT-BR, 28 categorias)
3. ✅ Treina BPE Tokenizer do zero
4. ✅ Pré-treina o Transformer (tiny: ~10M params ou small: ~30M params)
5. ✅ Testa geração de código
6. ✅ Fine-tuning LoRA com 125+ exemplos instrucionais
7. ✅ Download dos checkpoints via commit do Dataset

**Tempo estimado na T4 gratuita:** ~30min (tiny) a ~2h (small)

---
### ⚙️ Configuração necessária no Kaggle

Antes de executar, ative:
1. **Settings → Internet**: ✅ ON (precisa para clonar o repo)
2. **Settings → Accelerator**: GPU T4 x2 (ou P100)
3. **Persistence**: Os checkpoints ficam em `/kaggle/working/`. Após rodar, clique em **"Save Version"** (ou Commit) para salvar os outputs permanentemente.

---
### 📦 Saída (checkpoints gerados)

| Arquivo | Descrição |
|---------|-----------|
| `nine/data/nine1-base.pt` | Modelo pré-treinado (~44MB tiny, ~129MB small) |
| `nine/data/nine1-tok.json` | Tokenizer BPE treinado |
| `nine/data/nine1-instruct.pt` | LoRA após fine-tuning |
| `nine/data/corpus.bin` | Corpus tokenizado para treino |
| `nine/data/instruct.jsonl` | Dataset instrucional |

---
## 1. Setup — Verificação do Ambiente

In [ ]:
# (1) Verifica GPU e ambiente
import subprocess, sys

print('=== Ambiente Kaggle ===')
print(f'Python: {sys.version}')

try:
    out = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=10)
    print(out.stdout.split('\n')[0])
except:
    print('nvidia-smi: NAO ENCONTRADO (usando CPU)')

import torch
print(f'torch: {torch.__version__}')
print(f'CUDA disponivel: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device alvo: {DEVICE}')
print('[ok] Setup completo')

---
## 2. Clone o Repositório

In [ ]:
# (2) Clone do GitHub
# NOTA: Internet precisa estar ativada em Settings > Internet
import os, sys

REPO = 'https://github.com/Vtgamer998/nine-1'
WORKDIR = '/kaggle/working/nine-1'

if not os.path.exists(WORKDIR):
    !git clone {REPO} {WORKDIR}
    print('[ok] Repositorio clonado')
else:
    %cd {WORKDIR}
    !git pull
    print('[ok] Repositorio ja existe, atualizado')

%cd {WORKDIR}
sys.path.insert(0, '.')

# Verifica estrutura
print('\nEstrutura:')
for item in sorted(os.listdir('nine')):
    print(f'  nine/{item}')

---
## 3. Instalar Dependências

Kaggle já vem com torch, numpy por padrão. Só instalamos extras se necessário.

In [ ]:
# (3) Dependencias
import importlib

deps = {'numpy': 'numpy', 'torch': 'torch', 'json': None, 're': None}
missing = []
for name, pkg in deps.items():
    try:
        importlib.import_module(name)
    except ImportError:
        missing.append(pkg or name)

if missing:
    print(f'Instalando: {missing}')
    !pip install -q {' '.join(missing)}
else:
    print('Todas as dependencias estao OK')

# Verifica GPU suporta bfloat16
if torch.cuda.is_available():
    bf16 = torch.cuda.is_bf16_supported()
    print(f'bfloat16 suportado: {bf16}')
else:
    print('CPU mode: usando float32')

print('[ok] Dependencias OK')

---
## 4. Gerar Dataset Instrucional (125+ exemplos PT-BR)

Gera o arquivo `nine/data/instruct.jsonl` com 125+ exemplos de código cobrindo:
- Matemática, Estatística, Listas, Strings
- Ordenação, OOP, Algoritmos clássicos
- Conversões, Validação, Dicionários, Arquivos
- Geradores, Decorators, Recursão
- Compreensão de lista, Matrizes, Grafos, Conjuntos
- Context managers, Propriedades, Exceções, e mais

In [ ]:
# (4a) Gera dataset instrucional expandido
import importlib, importlib.util

spec = importlib.util.spec_from_file_location(
    "instruct_seed", "nine/data/instruct_seed.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
print('[ok] Dataset instrucional gerado')

In [ ]:
# (4b) Verifica o dataset
import json

with open('nine/data/instruct.jsonl') as f:
    lines = [json.loads(l) for l in f if l.strip()]

print(f'Total exemplos: {len(lines)}')
print(f'\nExemplos:')
for i, ex in enumerate(lines[:5]):
    print(f'  {i+1}. {ex["instruction"][:60]}...')
print(f'  ... e mais {len(lines)-5} exemplos')

# Tamanho total
total_chars = sum(len(ex['instruction']) + len(ex['output']) for ex in lines)
print(f'Total caracteres: {total_chars:,}')

---
## 5. Processar Seeds e Treinar BPE Tokenizer

Gera:
- `corpus.txt`: texto puro dos seeds
- `nine1-tok.json`: tokenizer BPE byte-level (GPT-2 style)
- `corpus.bin`: corpus tokenizado como uint16 para treino rápido

In [ ]:
# (5) Pre-processamento completo
!python -m nine.prep_data \
    --paths nine/data/seed \
    --out nine/data/corpus.txt \
    --train_bpe --vocab 4096 \
    --bin_out nine/data/corpus.bin \
    --tok_out nine/data/nine1-tok.json

print()
import os
for f in ['corpus.txt', 'corpus.bin', 'nine1-tok.json']:
    path = f'nine/data/{f}'
    if os.path.exists(path):
        sz = os.path.getsize(path)
        unit = 'MB' if sz > 1024*1024 else 'KB'
        val = sz / (1024*1024 if unit == 'MB' else 1024)
        print(f'  {f}: {val:.1f} {unit}')
    else:
        print(f'  {f}: AUSENTE!')
print('[ok] Dados preparados')

In [ ]:
# (5b) Teste do tokenizer - round-trip
import glob
from nine.tokenizer import BPETokenizer

bt = BPETokenizer.load('nine/data/nine1-tok.json')
print(f'Tokenizer: {len(bt)} tokens')

# Testa round-trip em todos os seeds
seed_files = sorted(glob.glob('nine/data/seed/seed_*.py'))
ok = fail = 0
for f in seed_files:
    text = open(f).read()
    ids = bt.encode(text)
    dec = bt.decode(ids)
    if dec == text:
        ok += 1
    else:
        fail += 1
        print(f'  FALHA: {f}')

print(f'Round-trip: {ok} OK, {fail} FAIL em {len(seed_files)} arquivos')
assert fail == 0, 'Tokenizer com falhas!'
print('[ok] Tokenizer OK')

---
## 6. Pré-Treino do Modelo

Duas opções de configuração:
- **Tiny** (~10M params, ~30min na T4): `n_layer=6, n_head=6, n_embd=384, block_size=256`
- **Small** (~30M params, ~2h na T4): `n_layer=10, n_head=8, n_embd=512, block_size=512`

**Arquitetura:** Transformer decoder-only com:
- RoPE (Rotary Position Embeddings)
- GQA (Grouped Query Attention: 6Q/3KV heads)
- RMSNorm (mais estável que LayerNorm)
- Gradient accumulation para batches maiores
- Cosine learning rate decay com warmup

In [ ]:
# (6) ESCOLHA A CONFIGURACAO:
# Descomente UMA das opcoes abaixo:

# --- OPCAO A: TINY (~10M params, rapido) ---
CONFIG = 'tiny'
TRAIN_ARGS = {
    '--vocab': 4096,
    '--block_size': 256,
    '--n_layer': 6,
    '--n_head': 6,
    '--n_embd': 384,
    '--batch_size': 64,
    '--max_iters': 2000,
    '--lr': '3e-4',
    '--warmup': 100,
    '--grad_accum_steps': 2,
}

# --- OPCAO B: SMALL (~30M params, melhor) ---
# Descomente para usar:
# CONFIG = 'small'
# TRAIN_ARGS = {
#     '--vocab': 4096,
#     '--block_size': 512,
#     '--n_layer': 10,
#     '--n_head': 8,
#     '--n_embd': 512,
#     '--batch_size': 32,
#     '--max_iters': 5000,
#     '--lr': '3e-4',
#     '--warmup': 200,
#     '--grad_accum_steps': 4,
# }

print(f'Configuracao: {CONFIG.upper()}')
print(f'Parametros: {TRAIN_ARGS}')
for k, v in TRAIN_ARGS.items():
    TRAIN_ARGS[k] = str(v)
print('[ok] Config definida')

In [ ]:
# (6a) Treino do modelo
import time

t0 = time.time()

cmd = f'python -m nine.train'
cmd += f' --data nine/data/corpus.bin'
cmd += f' --tok nine/data/nine1-tok.json'
cmd += f' --out nine/data/nine1-base.pt'
cmd += f' --device {DEVICE}'
for k, v in TRAIN_ARGS.items():
    cmd += f' {k} {v}'

print(f'Comando: {cmd}')
print(f'Iniciando treino...')
print()
!{cmd}

dt = time.time() - t0
print(f'\nTreino concluido em {dt/60:.1f} minutos')

ckpt_size = os.path.getsize('nine/data/nine1-base.pt')
print(f'Checkpoint salvo: {ckpt_size/1024/1024:.1f} MB')

---
## 7. Testar Geração de Código

Testa o modelo recém-treinado com prompts em PT-BR.

In [ ]:
# (7a) Teste: geração modo raw
!python -m nine.cli "def fibonacci(n):" \
    --ckpt nine/data/nine1-base.pt \
    --tok nine/data/nine1-tok.json \
    --tokens 100 --temp 0.8 --top_k 40

In [ ]:
# (7b) Teste: modo instruct
!python -m nine.cli "escreva uma funcao que valida email" \
    --mode instruct \
    --ckpt nine/data/nine1-base.pt \
    --tok nine/data/nine1-tok.json \
    --tokens 120 --temp 0.7 --top_k 30

In [ ]:
# (7c) Teste: fibonacci
!python -m nine.cli "crie uma classe Pilha em python" \
    --mode instruct \
    --ckpt nine/data/nine1-base.pt \
    --tok nine/data/nine1-tok.json \
    --tokens 150 --temp 0.7 --top_k 40

---
## 8. Fine-tuning LoRA (Opcional)

Aplica LoRA (Low-Rank Adaptation) nas projeções Q, K, V do modelo base.
Usa os 125+ exemplos instrucionais em PT-BR.

LoRA é eficiente: treina apenas ~0.3M parâmetros extras em vez de todo o modelo.

In [ ]:
# (8a) Fine-tune com LoRA
import time
t0 = time.time()

!python -m nine.finetune \
    --base nine/data/nine1-base.pt \
    --data nine/data/instruct.jsonl \
    --tok nine/data/nine1-tok.json \
    --out nine/data/nine1-instruct.pt \
    --lora_r 8 --lora_alpha 16 \
    --lora_target qkv \
    --batch_size 8 \
    --max_iters 500 --lr 2e-4 \
    --epochs 5 \
    --device {DEVICE}

print(f'LoRA fine-tuning concluido em {time.time()-t0:.0f}s')
lora_size = os.path.getsize('nine/data/nine1-instruct.pt')
print(f'LoRA salvo: {lora_size/1024:.1f} KB')

In [ ]:
# (8b) Testa com LoRA
!python -m nine.cli "escreva uma funcao fibonacci recursiva" \
    --mode instruct \
    --ckpt nine/data/nine1-base.pt \
    --lora nine/data/nine1-instruct.pt \
    --tok nine/data/nine1-tok.json \
    --tokens 100 --temp 0.6 --top_k 30

In [ ]:
# (8c) Mais testes com LoRA
prompts = [
    'crie uma classe ContaBancaria',
    'escreva uma funcao que calcula o fatorial',
    'escreva bubble sort em python',
    'crie uma classe Pilha',
]

for p in prompts:
    print(f'\n===== {p} =====')
    !python -m nine.cli "{p}" \
        --mode instruct \
        --ckpt nine/data/nine1-base.pt \
        --lora nine/data/nine1-instruct.pt \
        --tok nine/data/nine1-tok.json \
        --tokens 80 --temp 0.6 --top_k 30 2>&1 | tail -5
    print()

---
## 9. (Avançado) DPO — Direct Preference Optimization

Alinha o modelo com preferências humanas usando DPO.
Otimiza a política do modelo para preferir respostas "escolhidas" em vez de "rejeitadas".

**Nota:** Você precisa criar um dataset de preferências manualmente.
Um dataset DPO precisa de:
- `prompt`: a instrução
- `chosen`: a resposta boa (ex: código correto e limpo)
- `rejected`: a resposta ruim (ex: código bugado ou ineficiente)

In [ ]:
# (9a) Cria dataset DPO sintético (exemplo)
import json

dpo_data = [
    {
        'prompt': 'escreva fibonacci',
        'chosen': 'def fib(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a+b\n    return a',
        'rejected': 'def fib(n):\n    if n < 2:\n        return n\n    return fib(n-1) + fib(n-2)  # exponencial',
    },
    {
        'prompt': 'escreva uma funcao que calcula fatorial',
        'chosen': 'def fatorial(n):\n    if n <= 1:\n        return 1\n    return n * fatorial(n-1)',
        'rejected': 'def fatorial(n):\n    r = 1\n    for i in range(1, n+1):\n        r *= i\n    return r  # confuso',
    },
    {
        'prompt': 'escreva busca binaria',
        'chosen': 'def busca_binaria(lista, alvo):\n    esq, dir = 0, len(lista)-1\n    while esq <= dir:\n        meio = (esq+dir)//2\n        if lista[meio] == alvo:\n            return meio\n        elif lista[meio] < alvo:\n            esq = meio+1\n        else:\n            dir = meio-1\n    return -1',
        'rejected': 'def busca_binaria(lista, alvo):\n    return lista.index(alvo)  # O(n) nao binaria',
    },
]

with open('nine/data/preferences.jsonl', 'w') as f:
    for item in dpo_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f'Dataset DPO criado: {len(dpo_data)} exemplos')
print()
print('Para rodar DPO (quando implementado):')
print('  python -m nine.dpo_train')
print('    --base nine/data/nine1-instruct.pt')
print('    --data nine/data/preferences.jsonl')
print('    --tok nine/data/nine1-tok.json')
print('    --beta 0.1')
print('    --epochs 3')

---
## 10. (Opcional) Quantização GGUF

Converte o modelo para formato GGUF (GGML Universal Format) para rodar com `llama.cpp`.
Suporta quantização Q4_K_M (~4.5 bits por peso, boa qualidade).

**Requisito:** Necessita do `llama.cpp` compilado com suporte a GGUF.

In [ ]:
# (10) Quantizacao GGUF (experimental)
print('Quantizacao GGUF disponivel:')
print()
print('  python -m nine.quantize')
print('    --base nine/data/nine1-base.pt')
print('    --tok nine/data/nine1-tok.json')
print('    --out nine/data/nine1-q4.gguf')
print('    --qtype q4_k_m')
print()
print('Para usar com llama.cpp:')
print('  ./main -m nine/data/nine1-q4.gguf -p "# tarefa: escreva fibonacci" -n 100')
print()
print('(Implementacao em nine/quantize.py)')

---
## 11. Download dos Checkpoints

No Kaggle, para salvar os checkpoints permanentemente:
1. Clique em **"Save Version"** (ou **Commit**) no canto superior direito
2. Selecione **"Save & Run All"** ou **"Save Only"**
3. Os outputs serão salvos como uma nova versão do Dataset
4. Para baixar: vá em **Output** → faça download dos arquivos

Alternativa: você pode baixar manualmente com este notebook.

In [ ]:
# (11) Lista arquivos gerados
import os

print('=== Arquivos gerados ===')
print(f"{'Arquivo':<40} {'Tamanho':>10}")
print('-' * 52)

for path in [
    'nine/data/nine1-base.pt',       # Modelo pre-treinado
    'nine/data/nine1-tok.json',      # Tokenizer BPE
    'nine/data/nine1-instruct.pt',   # LoRA adapters
    'nine/data/corpus.bin',          # Dados tokenizados
    'nine/data/corpus.txt',          # Corpus texto puro
    'nine/data/instruct.jsonl',      # Dataset instrucional
    'nine/data/preferences.jsonl',   # Dataset DPO
]:
    if os.path.exists(path):
        sz = os.path.getsize(path)
        if sz > 1024 * 1024:
            label = f'{sz/1024/1024:.1f} MB'
        elif sz > 1024:
            label = f'{sz/1024:.1f} KB'
        else:
            label = f'{sz} B'
        print(f'{path:<40} {label:>10}')
    else:
        print(f'{path:<40} AUSENTE')

print('-' * 52)
print()
print('📌 Para baixar: Save Version > Output files')

---
## 12. Web UI (Opcional)

Inicia uma interface Gradio para interagir com o modelo pelo navegador.

**Atenção:** O Kaggle gera um link público (share) via Gradio, mas
pode ser bloqueado. Prefira rodar a Web UI localmente no Termux ou desktop.

In [ ]:
# (12) Web UI Gradio (descomente para rodar)
# NOTA: instala gradio se necessario
# !pip install -q gradio
# !python -m nine.webui \
#     --ckpt nine/data/nine1-base.pt \
#     --lora nine/data/nine1-instruct.pt \
#     --tok nine/data/nine1-tok.json \
#     --share --debug

---
## 13. Rodar Testes

Verifica se o modelo e componentes estão funcionando corretamente.

In [ ]:
# (13) Testes unitarios
!python tests/test_basic.py

---
## 14. Fuse: Carregar Modelo + LoRA

Funde o modelo base com os adaptadores LoRA em um único checkpoint.
Útil para deploy sem dependência de LoRA.

In [ ]:
# (14) Fusao do modelo base com LoRA
print('Fusao base + LoRA:')
print()
print('  python -m nine.fuse')
print('    --base nine/data/nine1-base.pt')
print('    --lora nine/data/nine1-instruct.pt')
print('    --tok nine/data/nine1-tok.json')
print('    --out nine/data/nine1-fused.pt')
print('    --validate')

---
✅ **Treinamento concluído!**

## Resumo dos resultados

| Item | Status |
|------|--------|
| Tokenizer BPE | `nine/data/nine1-tok.json` |
| Modelo pré-treinado | `nine/data/nine1-base.pt` |
| LoRA fine-tuning | `nine/data/nine1-instruct.pt` |
| Dataset instrucional | `nine/data/instruct.jsonl` (125+ exemplos) |
| Dataset DPO | `nine/data/preferences.jsonl` |

### Como usar fora do Kaggle

```bash
# No Termux ou desktop
python -m nine.cli "escreva fibonacci" \
    --ckpt nine/data/nine1-base.pt \
    --tok nine/data/nine1-tok.json \
    --tokens 80 --temp 0.7 --top_k 30

# Com LoRA
python -m nine.cli "crie uma classe Pilha" \
    --mode instruct \
    --ckpt nine/data/nine1-base.pt \
    --lora nine/data/nine1-instruct.pt \
    --tok nine/data/nine1-tok.json
```

### Links
- GitHub: https://github.com/Vtgamer998/nine-1
- Colab: notebooks/train_nine1.ipynb
- Kaggle: este notebook